In [1]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/src").parent))

from src.dataset_manager import DatasetManager
from src.model_config import ModelConfig

# Paths
new_data_dir = "/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/data_no_plurality_mixing"
lexicon_path = "/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json"

# Create the new data directory
Path(new_data_dir).mkdir(parents=True, exist_ok=True)

# Create a minimal ModelConfig (required fields only - values don't matter for data generation)
config = ModelConfig(
    n_layer=2,
    n_head=2,
    n_embd=128,
    batch_size=16,
    learning_rate=1e-4,
    warmup_steps=100,
    weight_decay=0.01,
    eval_steps=100
)

# Initialize DatasetManager with the new data directory
dm = DatasetManager(new_data_dir, config, lexicon_path)

# Generate data with NO plurality mixing (subject and object plurality must match)
print("Generating data with constrained plurality (subject and object must match)...")
train_df, test_df = dm.make_and_save_testing_and_training_data(
    test_size=0.1,
    random_seed=0
)

print(f"\nGenerated data:")
print(f"  Training examples: {len(train_df):,}")
print(f"  Test examples: {len(test_df):,}")
print(f"  Total examples: {len(train_df) + len(test_df):,}")

# Verify constraint
mismatched = train_df[train_df['subj_plural'] != train_df['obj_plural']]
if len(mismatched) == 0:
    print(f"  ✓ All examples have matching subject/object plurality")
else:
    print(f"  ✗ WARNING: {len(mismatched)} examples have mismatched plurality!")


/n/home06/drooryck/envs/codeswitching-py310/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Generating data with constrained plurality (subject and object must match)...

Generated data:
  Training examples: 295,872
  Test examples: 32,752
  Total examples: 328,624
  ✓ All examples have matching subject/object plurality


In [2]:
train_df

,input,target,lang,plural,subj_plural,obj_plural,subj,obj,verb
0,le chien voit le loup,le chien a vu le loup,fr,False,False,False,chien,loup,voit
1,les chiens voient les loups,les chiens ont vu les loups,fr,True,True,True,chien,loup,voit
2,le chien cherche le loup,le chien a cherché le loup,fr,False,False,False,chien,loup,cherche
3,les chiens cherchent les loups,les chiens ont cherché les loups,fr,True,True,True,chien,loup,cherche
4,le chien aide le loup,le chien a aidé le loup,fr,False,False,False,chien,loup,aide
...,...,...,...,...,...,...,...,...,...
295867,de spinnen spreken de hagedissen,de spinnen hebben de hagedissen gesproken,nl,True,True,True,spin,hagedis,spreekt
295868,de spin antwoordt de hagedis,de spin heeft de hagedis geantwoord,nl,False,False,False,spin,hagedis,antwoordt
295869,de spinnen antwoorden de hagedissen,de spinnen hebben de hagedissen geantwoord,nl,True,True,True,spin,hagedis,antwoordt
295870,de spin wacht de hagedis,de spin heeft de hagedis gewacht,nl,False,False,False,spin,hagedis,wacht


In [11]:
import pandas as pd
df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/dec16.0/runs/p50.00_run01/ablation_predictions.csv")
df[(df["ablation"] == "object") & (df["language"] == "fr")].iloc[:10]

,language,input,gold,prediction,ablation
3,fr,le chien mange de wolf,le chien a mangé le loup,le chien a mangé de wolf,object
7,fr,le chien mange de wolven,le chien a mangé les loups,le chien a mangé de wolven,object
11,fr,les chiens mangent de wolf,les chiens ont mangé le loup,les chiens ont mangé de wolf,object
15,fr,les chiens mangent de wolven,les chiens ont mangé les loups,les chiens ont mangé de wolven,object
19,fr,le chien aime de wolf,le chien a aimé le loup,le chien a aimé de aimé,object
23,fr,le chien aime de wolven,le chien a aimé les loups,le chien a aimé de aimé,object
27,fr,les chiens aiment de wolf,les chiens ont aimé le loup,les chiens ont aimé de aimé,object
31,fr,les chiens aiment de wolven,les chiens ont aimé les loups,les chiens ont aimé de aimé,object
35,fr,le chien casse de wolf,le chien a cassé le loup,le chien a cassé de cassé,object
39,fr,le chien casse de wolven,le chien a cassé les loups,le chien a cassé de cassé,object
